## Taints and tolerations — the node-side "go away"

`nodeSelector` and affinity are *pod-side* — "I want this kind of node." **Taints** are *node-side* — "stay away unless you've opted in." Together they let admins **reserve nodes** for specific workloads.

**Taint a node** — `<key>=<value>:<effect>`:

```bash
kubectl taint nodes gpu-node-1 dedicated=gpu:NoSchedule
```

| Effect | Meaning |
|---|---|
| `NoSchedule` | New non-tolerating pods won't schedule; existing stay. |
| `PreferNoSchedule` | Soft version — the scheduler avoids but won't refuse. |
| `NoExecute` | Won't schedule **and** evicts existing non-tolerating pods. |

**Tolerate on the pod:**

```yaml
tolerations:
  - { key: dedicated, operator: Equal, value: gpu, effect: NoSchedule }
```

Key point: **a toleration *unlocks* a taint, it doesn't *attract* the pod.** To actually steer a pod onto the tainted node, tolerate the taint **and** add `nodeSelector`/`nodeAffinity`. `operator: Exists` matches the key regardless of value; `Equal` (default) matches key + value.

**Built-in taints** the system applies: `node.kubernetes.io/not-ready`, `.../unreachable`, `.../memory-pressure`, `.../disk-pressure`, `.../unschedulable` (cordon), and `node-role.kubernetes.io/control-plane:NoSchedule` on masters. Every pod implicitly tolerates `not-ready`/`unreachable` for **300s** so a blip doesn't evict; `tolerationSeconds` tunes that.

Remove a taint with a trailing minus: `kubectl taint nodes gpu-node-1 dedicated=gpu:NoSchedule-`. The CKA wants you quick with both `kubectl taint` and hand-writing tolerations. On our map these are the **taints** and **tolerations** chips — the node saying "go away," the pod saying "I'm allowed."